In [ ]:
!pip -q install openai-whisper
!pip -q install groq
!pip -q install edge-tts
!pip -q install gradio
!pip -q install python-dotenv
!pip -q install ffmpeg-python
!pip -q install nest_asyncio

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
^C
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 78, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/__init__.py", line 114, in create_command
    module = importlib.import_module(module_path)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap

In [ ]:
import whisper
import gradio as gr
import asyncio
import edge_tts
import tempfile
import os
from groq import Groq
import nest_asyncio
nest_asyncio.apply()

In [ ]:
print("Loading Whisper model...")
model = whisper.load_model("base")
print("Whisper model loaded successfully!")

Loading Whisper model...
Whisper model loaded successfully!


In [ ]:
GROQ_API_KEY = "your api"
client = Groq(api_key=GROQ_API_KEY)
print("Groq API Connected Successfully!")

Groq API Connected Successfully!


In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Introduce yourself in one sentence."
        }
    ]
)
print(response.choices[0].message.content)

I'm an artificial intelligence model designed to provide information and assist with tasks to the best of my abilities, and I'm here to help answer your questions and provide assistance.


In [ ]:
async def text_to_speech(text):
    output_file = "response.mp3"
    communicate = edge_tts.Communicate(
        text=text,
        voice="en-US-AriaNeural"
    )
    await communicate.save(output_file)
    return output_file

In [ ]:
import asyncio

audio = await text_to_speech("Hello Hamza! Welcome to your AI Voice Assistant.")
print(audio)

response.mp3


In [ ]:
def speech_to_text(audio_path):
    result = model.transcribe(audio_path)
    return result["text"]

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving WhatsApp Ptt 2026-07-11 at 4.26.07 AM.ogg to WhatsApp Ptt 2026-07-11 at 4.26.07 AM.ogg


In [ ]:
audio_file = list(uploaded.keys())[0]
text = speech_to_text(audio_file)
print("Recognized Text:")
print(text)

Recognized Text:
 What is AI?


In [ ]:
def ask_ai(user_text):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful AI Voice Assistant. Give short and clear answers."
            },
            {
                "role": "user",
                "content": user_text
            }
        ]
    )
    return response.choices[0].message.content

In [ ]:
reply = ask_ai("What is Artificial Intelligence?")
print(reply)

Artificial Intelligence (AI) is a technology that allows machines to think, learn, and act like humans. It uses algorithms and data to make decisions, solve problems, and perform tasks automatically.


In [ ]:
def voice_assistant(audio):

    user_text = speech_to_text(audio)

    ai_response = ask_ai(user_text)

    # Use asyncio.run now that nest_asyncio is applied
    audio_response = asyncio.run(text_to_speech(ai_response))
    return user_text, ai_response, audio_response

In [ ]:
user_text, ai_text, audio_file = voice_assistant(audio_file) # Removed await
print("User:")
print(user_text)
print()
print("Assistant:")
print(ai_text)
print()
print(audio_file)

User:
 What is AI?

Assistant:
AI stands for Artificial Intelligence. It's a computer system that can think, learn, and act like a human.

response.mp3


In [ ]:
interface = gr.Interface(
    fn=voice_assistant,
    inputs=gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="🎤 Speak Here"
    ),
    outputs=[
        gr.Textbox(label="📝 Recognized Speech"),
        gr.Textbox(label="🤖 AI Response"),
        gr.Audio(label="🔊 Voice Response")
    ],
    title="🎙 AI Voice Assistant",
    description="Speak into the microphone and the AI will answer using voice."
)

In [ ]:
import shutil
from google.colab import files
import os
import tempfile # Added for text_to_speech output file handling

# Define contents of app.py
app_py_content = """
import whisper
import gradio as gr
import asyncio
import edge_tts
import tempfile
import os
from groq import Groq
import nest_asyncio

nest_asyncio.apply()

print("Loading Whisper model...")
model = whisper.load_model("base")
print("Whisper model loaded successfully!")

# NOTE: For a production app, it's recommended to load the API key from environment variables.
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "******************************")
client = Groq(api_key=GROQ_API_KEY)
print("Groq API Connected Successfully!")


async def text_to_speech(text):
    output_file = os.path.join(tempfile.gettempdir(), "response.mp3") # Use tempfile for output
    communicate = edge_tts.Communicate(
        text=text,
        voice="en-US-AriaNeural"
    )
    await communicate.save(output_file)
    return output_file

def speech_to_text(audio_path):
    result = model.transcribe(audio_path)
    return result["text"]

def ask_ai(user_text):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful AI Voice Assistant. Give short and clear answers."
            },
            {
                "role": "user",
                "content": user_text
            }
        ]
    )
    return response.choices[0].message.content

def voice_assistant(audio):
    user_text = speech_to_text(audio)
    ai_response = ask_ai(user_text)
    audio_response = asyncio.run(text_to_speech(ai_response))
    return user_text, ai_response, audio_response

interface = gr.Interface(
    fn=voice_assistant,
    inputs=gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="🎤 Speak Here"
    ),
    outputs=[
        gr.Textbox(label="📝 Recognized Speech"),
        gr.Textbox(label="🤖 AI Response"),
        gr.Audio(label="🔊 Voice Response")
    ],
    title="🎙 AI Voice Assistant",
    description="Speak into the microphone and the AI will answer using voice."
)

if __name__ == "__main__":
    interface.launch(debug=True)
"""

# Define contents of requirements.txt
requirements_txt_content = """
openai-whisper
groq
edge-tts
gradio
python-dotenv
ffmpeg-python
nest_asyncio
"""

# Define contents of README.md
readme_md_content = """
# AI Voice Assistant

This is a simple AI Voice Assistant built with Gradio, OpenAI Whisper for Speech-to-Text,
Groq for AI language model interaction (Llama-3.3-70b-versatile), and Edge TTS for Text-to-Speech.

## Setup and Run

1.  **Install dependencies**:
    ```bash
    pip install -r requirements.txt
    ```
2.  **Set your Groq API Key**:
    Create a `.env` file in the root directory and add your Groq API key:
    ```
    GROQ_API_KEY="your_groq_api_key_here"
    ```
    (Note: In this Colab, the key is hardcoded for demonstration, but for production, use environment variables.)
3.  **Run the application**:
    ```bash
    python app.py
    ```
    This will launch the Gradio interface in your browser.
"""

# Create project folder
shutil.rmtree("AI-Voice-Assistant", ignore_errors=True)
os.makedirs("AI-Voice-Assistant", exist_ok=True)

# Write contents to files
with open("app.py", "w") as f:
    f.write(app_py_content)
with open("requirements.txt", "w") as f:
    f.write(requirements_txt_content)
with open("README.md", "w") as f:
    f.write(readme_md_content)

# Copy files (now they exist) into the project folder
shutil.copy("app.py", "AI-Voice-Assistant/app.py")
shutil.copy("requirements.txt", "AI-Voice-Assistant/requirements.txt")
shutil.copy("README.md", "AI-Voice-Assistant/README.md")

# Zip the folder
shutil.make_archive("AI-Voice-Assistant", "zip", "AI-Voice-Assistant")

# Download
files.download("AI-Voice-Assistant.zip")

# Clean up local files to avoid clutter
os.remove("app.py")
os.remove("requirements.txt")
os.remove("README.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>